# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the FAIR\(^2\) dataset using the `mlcroissant` library. All entities (record sets, fields, etc.) are referenced by their `@id` for clarity and reproducibility.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata object and print high-level description
md = dataset.metadata
print(f"{md.name}: {md.description}\n")
print(f"Published: {md.datePublished}")
print(f"License: {md.license}")
print(f"Version: {md.version}")
print(f"Keywords: {md.keywords}")

## 2. Data Overview
Review available record sets and their structure using their `@id`. This step lets us know which parts of the dataset we can work with.

In [ ]:
# Get the list of available record sets and their IDs
record_sets = list(dataset.record_sets.keys())
print(f"Record sets available (by @id):\n{record_sets}\n")

# For this dataset, print the fields available in each record set
for record_set_id in record_sets:
    record_set = dataset.record_sets[record_set_id]
    print(f'---\nRecordSet @id: {record_set_id}')
    print(f'Name: {record_set.name}')
    print(f'Description: {getattr(record_set, "description", "No description provided.")}')
    print('Fields:')
    for field_id, field in record_set.fields.items():
        field_name = getattr(field, 'name', field_id)
        print(f"    Field @id: {field_id} \t(name: {field_name}, type: {getattr(field, 'data_type', 'unknown')})")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use record set and field `@id`s from the previous overview.

In [ ]:
# To proceed, we will load each record set to pandas DataFrame and inspect columns (using @id as keys)

# List of record set @id(s) (update if more are added)
record_sets_ids = record_sets  # already listed from above
dataframes = {}

for record_set_id in record_sets_ids:
    # Each record yields a dict with field @id as key
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded record set {record_set_id}, {len(df)} rows, columns by @id:\n{df.columns.tolist()}\n")

# For demonstration, we'll focus on the first record set
current_rs_id = record_sets_ids[0]
print(f"First 5 rows of record set {current_rs_id}:")
dataframes[current_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records, normalizing numeric fields, and grouping. All references use record set and field `@id`s.

In [ ]:
# Choose a numerical field by @id; inspect available numeric columns
df = dataframes[current_rs_id]
print("Available columns (by field @id):", df.columns.tolist())

# Let's do basic EDA on a numeric field. Below is a guess based on common proteomic datasets: choose a field with MW (molecular weight) or abundance
# Replace with the correct @id for your dataset. Example:
for col in df.columns:
    if 'MW' in col or 'abundance' in col or 'count' in col.lower() or df[col].dtype in [float, int]:
        print(f"Examining numeric field: {col}")
        numeric_field_id = col
        break
else:
    # Fall back to the first numeric-looking column
    for col in df.columns:
        try:
            _ = pd.to_numeric(df[col])
            numeric_field_id = col
            print(f"(Fallback) Using: {col}")
            break
        except:
            continue

field_values = pd.to_numeric(df[numeric_field_id], errors='coerce')
threshold = field_values.mean()
filtered_df = df[field_values > threshold]
print(f"Filtered records with field {numeric_field_id} > mean ({threshold:.2f})\nN={len(filtered_df)}")

# Normalize the numeric field (z-score)
filtered_df[f"{numeric_field_id}_normalized"] = (pd.to_numeric(filtered_df[numeric_field_id], errors='coerce') - field_values.mean()) / field_values.std()
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try to group by another categorical field (e.g., description, accession etc. by @id)
group_by_field = None
for col in df.columns:
    if col != numeric_field_id and df[col].nunique() > 1 and df[col].nunique() < len(df) / 2:
        group_by_field = col
        break
if group_by_field is not None:
    grouped_df = filtered_df.groupby(group_by_field)[numeric_field_id].mean().to_frame()
    print(f"Grouped data by {group_by_field} (mean {numeric_field_id}):")
    print(grouped_df.head())
else:
    print("No suitable group field found.")

## 5. Visualization
Visualize the distribution of the numeric field and a basic grouped summary. You can adjust the field `@id`s as needed for exploration.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of normalized numeric field
plt.figure(figsize=(7,4))
sns.histplot(filtered_df[f"{numeric_field_id}_normalized"].dropna(), bins=30, kde=True)
plt.title(f"Distribution of normalized {numeric_field_id}")
plt.xlabel(f"{numeric_field_id} (normalized; @id)")
plt.ylabel("Count")
plt.show()

# Grouped bar chart if group_by_field is available
if group_by_field is not None:
    grouped_df = grouped_df.sort_values(by=numeric_field_id, ascending=False).head(10)
    plt.figure(figsize=(8,4))
    sns.barplot(y=grouped_df.index, x=grouped_df[numeric_field_id], palette='viridis')
    plt.title(f"Top 10 {group_by_field} by mean {numeric_field_id}")
    plt.xlabel(f"Mean {numeric_field_id}")
    plt.ylabel(f"{group_by_field} (@id)")
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we've demonstrated how to load and explore the FAIR\(^2\) dataset using `mlcroissant`, referencing all dataset entities and fields by their `@id`. You can continue your analysis by substituting field `@id`s for your biological or statistical questions, integrating additional data transformations, or exporting processed DataFrames for downstream analyses.